In [ ]:
# google-api-python-client
# pyasn1
# uritemplate
# rsa
# pyasn1-modules
# httplib2
# googleapis-common-protos
# google-auth
# google-auth-httplib2
# google-api-core

In [1]:
from googleapiclient.discovery import build
import logging
import json

In [2]:
# Set API key
API_KEY = ''

# Set the playlist ID
PL_ID = 'PLKnIA16_RmvbAlyx4_rdtR66B7EHX5k3z'

In [15]:
def get_playlist_videos(playlist_id):
    youtube = build('youtube', 'v3', developerKey=API_KEY)
    request = youtube.playlistItems().list(
        part='snippet',
        maxResults=50,
        playlistId=playlist_id
    )
    videos = []
    while request is not None:
        response = request.execute()
        videos += response['items']
        request = youtube.playlistItems().list_next(request, response)
    
    video_details = []
    for video in videos:
        video_id = video['snippet']['resourceId']['videoId']
        title = video['snippet']['title']
        video_details.append({'title': title, 'video_id': video_id})
        
    return video_details


In [16]:
# Store playlist video details in json file
with open('playlist.json', 'w') as pl:
    pl_dict = {'items': []}
    
    pl_dict['items'] = get_playlist_videos(PL_ID)
    
    json.dump(pl_dict, pl, indent=2)

In [3]:
def get_video_details(video_id: list[str]):
    youtube = build('youtube', 'v3', developerKey=API_KEY)

    # Dictanory to store video details
    video_details_dict = {'video_deatils': []}

    logging.basicConfig(filename='./logs/get_video_details.log',
                        level=logging.ERROR,
                        format='%(asctime)s %(name)s %(levelname)s:\n%(message)s')
    for _id in video_id:
        request = youtube.videos().list(
            part='snippet',
            id=_id
        )
        response = request.execute()
        try:
            video = response['items'][0]
        except IndexError as e:
            # Log the error
            logging.error(f'Response gives IndexError at video_id={_id}')

            # Append the data as deleted video.
            video_details_dict['video_deatils'].append({
                'title': 'Deleted Video',
                'description': 'Deleted Video',
                'video_id': _id
            })
            continue
        title = video['snippet']['title']
        description = video['snippet']['description']

        # Append the details
        video_details_dict['video_deatils'].append({
            'title': title,
            'description': description,
            'video_id': _id
        })

    return video_details_dict


In [5]:
# Get video ids from playlist data
with open('./playlist.json', 'r') as pl:
    data = json.load(pl)['items']
    
    # Extract all ids from playlist.json file
    id_list = [i['video_id'] for i in data]

    # Create video details dictonary
    video_details_dict = get_video_details(id_list)

    # Create json file to store video details
    with open('./video_details2.json', 'w') as vd:
        # Dump the video details dict in json file
        json.dump(video_details_dict, vd, indent=2)

In [1]:
import json

with open('data/playlist_enhanced.json') as f:
    pl_enh = json.load(f)['items']

In [6]:
import pandas as pd

pd.DataFrame.from_dict(pl_enh).applymap(lambda x: pd.NA if x == '<NAN>' else x).isnull().sum()

title           0
video_id        0
isPaid          0
isSolutions     0
isSession       0
sessionNo      69
taskNo         78
sessionName     0
dtype: int64